In [ ]:
# Select the best model
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
best_model = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"F1-Score: {results_df.loc[results_df['Model'] == best_model_name, 'F1-Score'].values[0]:.4f}")

# Example: Make predictions on test set
if best_model:
    y_pred_best = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)
    
    # Create predictions dataframe
    predictions_df = pd.DataFrame({
        'Predicted_Label': y_pred_best,
        'Prediction': ['Placed' if p == 1 else 'Not Placed' for p in y_pred_best],
        'Probability_Not_Placed': y_pred_proba[:, 0],
        'Probability_Placed': y_pred_proba[:, 1]
    })
    
    print(f"\nSample Predictions (first 10):")
    print(predictions_df.head(10).to_string(index=False))
    
    print(f"\nPrediction Summary:")
    print(f"Total Predictions: {len(predictions_df)}")
    print(f"Predicted Placed: {(predictions_df['Prediction'] == 'Placed').sum()}")
    print(f"Predicted Not Placed: {(predictions_df['Prediction'] == 'Not Placed').sum()}")

## 11. Make Predictions on New Data

In [ ]:
if trained_models:
    # Feature importance from tree-based models
    tree_models = {
        'Random Forest': trained_models.get('Random Forest'),
        'Gradient Boosting': trained_models.get('Gradient Boosting')
    }
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for (name, model), ax in zip(tree_models.items(), axes):
        if model is not None and hasattr(model, 'feature_importances_'):
            # Get feature importance
            importance = model.feature_importances_
            feature_names = X.columns
            
            # Create DataFrame for visualization
            importance_df = pd.DataFrame({
                'Feature': feature_names,
                'Importance': importance
            }).sort_values('Importance', ascending=False).head(10)
            
            # Plot
            importance_df.plot(x='Feature', y='Importance', kind='barh', ax=ax, legend=False, color='#FF6B6B')
            ax.set_title(f'{name} - Top 10 Features', fontweight='bold')
            ax.set_xlabel('Importance')
            
            print(f"\n{name} - Top Features:")
            print(importance_df.to_string(index=False))
    
    plt.tight_layout()
    plt.show()

## 10. Feature Importance Analysis

In [ ]:
if results_df is not None and len(results_df) > 0:
    # Plot model comparison
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        results_df.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False, color='#4ECDC4')
        ax.set_title(f'{metric} Comparison', fontweight='bold')
        ax.set_ylabel(metric)
        ax.set_xlabel('')
        ax.set_ylim([0, 1])
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=2)
    
    plt.tight_layout()
    plt.show()

## 9. Visualize Model Performance

In [ ]:
if trained_models:
    results = []
    
    for name, model in trained_models.items():
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })
        
        print(f"\n{'='*50}")
        print(f"Model: {name}")
        print(f"{'='*50}")
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        print(f"\nConfusion Matrix:")
        print(f"  Not Placed: TN={cm[0][0]}, FP={cm[0][1]}")
        print(f"  Placed:     FN={cm[1][0]}, TP={cm[1][1]}")
        
        # Classification Report
        print(f"\nDetailed Classification Report:")
        print(classification_report(y_test, y_pred, target_names=['Not Placed', 'Placed']))
    
    # Comparison DataFrame
    results_df = pd.DataFrame(results)
    print(f"\n{'='*50}")
    print("Model Comparison:")
    print(f"{'='*50}")
    print(results_df.to_string(index=False))

## 8. Evaluate Model Performance

In [ ]:
if X_train is not None:
    # Initialize models
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    }
    
    # Train models
    trained_models = {}
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = model
        print(f"✓ {name} trained successfully\n")
    
    print("All models trained!")

## 7. Train Classification Models

In [ ]:
if X is not None and y is not None:
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set size: {X_train.shape[0]}")
    print(f"Testing set size: {X_test.shape[0]}")
    print(f"Training set target distribution:\n{y_train.value_counts(normalize=True)}")
    print(f"\nTesting set target distribution:\n{y_test.value_counts(normalize=True)}")

## 6. Split Data into Training and Testing Sets

In [ ]:
if df_clean is not None:
    # Create visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Target distribution
    y.value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
    axes[0].set_title('Target Distribution (Placement Status)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Placed (0=No, 1=Yes)')
    axes[0].set_ylabel('Count')
    
    # Placement percentage
    placement_pct = (y.value_counts(normalize=True) * 100).round(2)
    axes[1].pie(placement_pct.values, labels=['Not Placed', 'Placed'], autopct='%1.1f%%', 
                colors=['#FF6B6B', '#4ECDC4'], startangle=90)
    axes[1].set_title('Placement Percentage', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Feature correlations
    numeric_features = X.select_dtypes(include=[np.number]).columns
    if len(numeric_features) > 1:
        correlation_matrix = X[numeric_features].corr()
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm')
        plt.title('Feature Correlations', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 5. Exploratory Data Analysis

In [ ]:
if df_clean is not None and 'placed' in df_clean.columns:
    # Identify target and features
    target = 'placed'
    
    # Separate features and target
    X = df_clean.drop(columns=[target])
    y = df_clean[target]
    
    # Drop non-predictive columns (like id)
    cols_to_drop = [col for col in X.columns if col in ['id', 'Unnamed: 0', 'idx']]
    X = X.drop(columns=cols_to_drop, errors='ignore')
    
    # Encode categorical features
    categorical_features = X.select_dtypes(include=['object']).columns
    label_encoders = {}
    
    for col in categorical_features:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    print(f"Features shape: {X.shape}")
    print(f"Feature names: {list(X.columns)}")
    print(f"Target distribution:")
    print(y.value_counts())

## 4. Feature Engineering

In [ ]:
if df is not None:
    # Create a copy for preprocessing
    df_clean = df.copy()
    
    # Handle missing values
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    
    # Fill missing numeric values with mean
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mean(), inplace=True)
    
    # Fill missing categorical values with mode
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    
    print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")
    print(f"Cleaned dataset shape: {df_clean.shape}")

## 3. Data Preprocessing and Cleaning

In [ ]:
# Load the dataset
data_path = Path("../data/train.csv")

if not data_path.exists():
    print(f"⚠️  Data file not found at {data_path}")
    print("Please add your training data to the data/ folder")
    df = None
else:
    df = pd.read_csv(data_path)
    print(f"Dataset shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nDataset info:")
    print(df.info())
    print(f"\nMissing values:")
    print(df.isnull().sum())

## 2. Load and Explore the Dataset

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

print("✅ All libraries imported successfully")

## 1. Import Required Libraries

# Campus Placement Prediction using Machine Learning

This notebook implements a complete machine learning pipeline to predict campus placements for students.